<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/13-foundation-models-multimodal-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Foundation Models and Multimodal Learning** {#foundation-models-multimodal-learning}

A foundation model is pretrained on broad data and designed to support many downstream uses through prompting, retrieval, adapters, fine-tuning, or composition with other systems. Its importance comes from **reuse and leverage**: one upstream representation, dataset decision, interface, or defect can influence many deployments. Multimodal learning extends this reuse across text, images, audio, video, depth, sensor streams, and structured records.

The term does not mean “any large neural network.” The [Stanford CRFM report](https://arxiv.org/abs/2108.07258) emphasizes broad training and adaptation while also describing foundation models as incomplete sociotechnical systems. A checkpoint without documented data, adaptation contracts, evaluations, and monitoring is not a complete foundation-model program.

![A foundation model lifecycle from broad data to many adapted deployments and feedback.](assets/dl13-foundation-lifecycle.svg){fig-align="center" width="76%" fig-alt="Broad governed data flows into pretraining, adaptation, and many deployments, with monitoring feedback returning to the training system."}

*Original synthesis based on the lifecycle and downstream leverage discussed in the [CRFM foundation-model report](https://arxiv.org/abs/2108.07258).* 

This chapter uses scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), CC BY 4.0. Each real 8×8 image is paired with a **constructed and fully disclosed** English class description such as `a handwritten digit three` and two coarse attributes such as `odd, low`. These captions are not original UCI annotations. They create a small, auditable image-text mechanism study; conclusions are not presented as VLM benchmark results.

![Real digit images paired with the controlled text and attribute modalities used in the experiments.](assets/dl13-controlled-pairs.svg){fig-align="center" width="76%" fig-alt="Ten handwritten digit images paired with English digit names and coarse parity and magnitude attributes."}

*Original data visualization. Images come from the UCI-derived dataset; text and attributes are constructed for this chapter and are available for every split by design.*

<details>
<summary><strong>PyTorch: establish the shared image-text dataset and provenance contract</strong></summary>

```python
import copy
import hashlib
import math
import random
from collections import Counter

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1313):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1313, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1313,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]

digit_names = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
caption_templates = [
    "a handwritten digit {}",
    "an image of the number {}",
    "the optical character {}",
]
all_template_texts = [template.format(name) for name in digit_names for template in caption_templates]
vocabulary = {"<pad>": 0, "<unk>": 1}
for token in sorted({token for text in all_template_texts for token in text.split()}):
    vocabulary[token] = len(vocabulary)


def encode_texts(texts, max_length=7):
    ids = torch.zeros(len(texts), max_length, dtype=torch.long)
    mask = torch.zeros(len(texts), max_length, dtype=torch.bool)
    for row, text in enumerate(texts):
        tokens = [vocabulary.get(token, vocabulary["<unk>"]) for token in text.split()][:max_length]
        ids[row, :len(tokens)] = torch.tensor(tokens)
        mask[row, :len(tokens)] = True
    return ids, mask


def captions_for(indices, labels):
    return [caption_templates[int(index) % len(caption_templates)].format(digit_names[int(label)])
            for index, label in zip(indices, labels)]


train_text_ids, train_text_mask = encode_texts(captions_for(train_idx, train_y))
val_text_ids, val_text_mask = encode_texts(captions_for(val_idx, val_y))
test_text_ids, test_text_mask = encode_texts(captions_for(test_idx, test_y))


def hint_features(labels):
    # These are intentionally supplied side information, not hidden targets.
    return torch.stack([(labels % 2 == 0).float(), (labels >= 5).float()], dim=1)


train_hints, val_hints, test_hints = hint_features(train_y), hint_features(val_y), hint_features(test_y)


class ImageEncoder(nn.Module):
    def __init__(self, output_dim=32, hidden_dim=96):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, images):
        return self.network(images)


@torch.no_grad()
def classifier_accuracy(model, images, labels):
    model.eval()
    output = model(images)
    logits = output[0] if isinstance(output, tuple) else output
    return float((logits.argmax(1) == labels).float().mean())


assert all_images.shape == (1797, 1, 8, 8)
assert len(set(train_idx) & set(test_idx)) == 0
assert train_text_ids.shape == (1257, 7)
assert vocabulary["<pad>"] == 0
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "vocabulary size": len(vocabulary),
    "example pair": (captions_for(train_idx[:1], train_y[:1])[0], int(train_y[0])),
    "constructed modalities": ["image", "class description", "parity/magnitude hint"],
})
```

</details>

The split unit is the original image index. Text is generated only after splitting, so alternate templates for one image cannot leak across train and test. The coarse hints are treated as an available input modality in fusion experiments; they are never described as naturally collected metadata.

### **What Is a Foundation Model?** {#what-is-a-foundation-model}

Three properties distinguish the foundation-model pattern:

1. **Broad pretraining:** the upstream objective spans enough data, tasks, or modalities to learn reusable structure.
2. **Adaptability:** users can specify new behavior through prompts, retrieval, heads, PEFT, fine-tuning, tools, or structured outputs.
3. **Downstream leverage:** many systems inherit the same representations and therefore the same capabilities, biases, and vulnerabilities.

Scale is often an enabler, but breadth and interfaces matter more than a parameter threshold. A small domain model pretrained across many hospitals and adapted to multiple clinical tasks can follow the pattern; a huge classifier trained for one immutable label set may not. “General-purpose” is also not “universally competent.” Capability depends on the training support, input interface, context, evaluation, and deployment environment.

A useful abstraction is

$$
z=F_{\theta}(x, c),\qquad
\hat y=A_{\phi}(z, q),
$$

where $F_{\theta}$ is a broadly pretrained base, $x$ is observed data, $c$ is context or another modality, $A_{\phi}$ is an adaptation/interface layer, and $q$ specifies a downstream request. The separation is conceptual rather than mandatory: in-context learning may keep $\theta$ fixed, full fine-tuning changes it, and retrieval changes $c$ at inference.

The same leverage creates correlated risk. A representation shortcut can propagate into every adapted classifier. Memorized private data can be exposed through several interfaces. A tokenizer or image preprocessing defect can silently affect unrelated applications. Foundation-model evaluation must therefore include both **upstream properties** and **downstream use-specific evidence**; one average benchmark cannot certify all adaptations.

### **Scaling Laws and Compute-Optimal Training** {#scaling-laws-compute-optimal-training}

Scaling laws are empirical regularities relating loss to model size $N$, data exposure $D$, and training compute $C$. A common separable form is

$$
L(N,D)\approx L_{\infty}+aN^{-\alpha}+bD^{-\beta},
$$

where $L_{\infty}$ is an irreducible floor for the chosen distribution and objective. The constants and exponents are fitted within a particular regime; they are not universal physical laws. [Kaplan et al.](https://arxiv.org/abs/2001.08361) documented power-law behavior for language-model loss. [Hoffmann et al.](https://arxiv.org/abs/2203.15556) later used IsoFLOP analyses and found that, under their setting, compute-optimal model size and token count should grow roughly together.

For dense Transformer training, a rough order-of-magnitude proxy is $C\approx 6ND$ FLOPs. Under fixed $C$, increasing $N$ leaves fewer token updates $D\approx C/(6N)$. A model can therefore be **undertrained**: parameter capacity is purchased but not exposed to enough data. Deployment cost also matters; two models with similar pretraining compute can have very different inference memory and latency.

![Conceptual allocation of fixed compute between model size and data exposure.](assets/dl13-scaling-laws.svg){fig-align="center" width="74%" fig-alt="A conceptual IsoFLOP graph shows a balanced frontier between model parameter count and training token count under fixed compute."}

*Original conceptual diagram based on the IsoFLOP reasoning in [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556). It does not reproduce fitted paper coefficients.*

The small experiment below changes hidden width and examples per class while keeping the objective and optimizer fixed. It is too small to estimate a general scaling law; its purpose is to show the measurement table required before fitting one.

<details>
<summary><strong>PyTorch: build a small model-data scaling table</strong></summary>

```python
class WidthClassifier(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, width), nn.ReLU(), nn.Linear(width, 10)
        )

    def forward(self, images):
        return self.network(images)


def balanced_subset(labels, per_class, seed):
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in range(10):
        candidates = torch.where(labels == label)[0]
        chosen.append(candidates[torch.randperm(len(candidates), generator=generator)[:per_class]])
    return torch.cat(chosen)


def fit_scaling_point(width, per_class, epochs=35):
    seed_everything(1300 + width + per_class)
    subset = balanced_subset(train_y, per_class, seed=1300 + per_class)
    model = WidthClassifier(width)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loader = DataLoader(
        TensorDataset(train_x[subset], train_y[subset]), batch_size=64,
        shuffle=True, generator=torch.Generator().manual_seed(1300 + per_class),
    )
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        validation_loss = float(F.cross_entropy(model(val_x), val_y))
        validation_accuracy = classifier_accuracy(model, val_x, val_y)
    parameters = sum(p.numel() for p in model.parameters())
    compute_proxy = parameters * len(subset) * epochs
    return {"width": width, "examples": len(subset), "parameters": parameters,
            "compute_proxy": compute_proxy, "val_loss": validation_loss,
            "val_accuracy": validation_accuracy}


scaling_rows = [fit_scaling_point(width, per_class)
                for width in (16, 32, 64) for per_class in (10, 40)]
for row in scaling_rows:
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})

assert len(scaling_rows) == 6
assert all(math.isfinite(row["val_loss"]) for row in scaling_rows)
assert len({row["compute_proxy"] for row in scaling_rows}) == 6
```

</details>

A credible scaling study trains several points at the same compute budget, tunes learning schedules fairly, repeats seeds, and fits held-out predictions. Accuracy is often unsuitable because it saturates and is discontinuous; pretraining cross-entropy is more informative. Extrapolation beyond observed scales must include uncertainty, and a scaling curve says nothing by itself about safety, data rights, factuality, or downstream utility.

### **Training Data Curation and Governance** {#training-data-curation-governance}

At foundation-model scale, data is part of the algorithm. Source mixture, language and geography coverage, filtering thresholds, deduplication, temporal cutoff, annotation policy, licensing, personal information, and opt-out mechanisms determine what the model can learn and whose errors are hidden. [DataComp](https://arxiv.org/abs/2304.14108) demonstrated that dataset design can be studied under standardized training compute rather than treated as an undocumented preprocessing step.

![A curation pipeline from acquisition and documentation through filtering, deduplication, and audit.](assets/dl13-data-curation.svg){fig-align="center" width="76%" fig-alt="A five-stage data curation pipeline includes provenance, documentation, filtering, deduplication, and subgroup and contamination audits, with feedback."}

*Original synthesis informed by [DataComp](https://arxiv.org/abs/2304.14108) and the documentation principles in [Datasheets for Datasets](https://arxiv.org/abs/1803.09010).* 

Deduplication serves several purposes. It prevents popular or copied examples from receiving unintended weight, reduces train-test contamination, and makes evaluation less optimistic. Exact hashing catches identical bytes; near-duplicate detection requires perceptual, semantic, or document-level grouping. The split must happen by duplicate group, source document, speaker, patient, or video rather than by isolated row when those entities share information.

Multimodal pairs add alignment quality. A fluent caption may describe only a small part of an image; alt text may contain filenames or SEO text; temporal audio may be shifted from video; synthetic captions can amplify a teacher's errors. Filtering aggressively can also erase dialects, minority contexts, difficult examples, and safety-relevant data. Governance therefore needs records of what was removed as well as what was kept.

<details>
<summary><strong>Python: audit duplicate images and corrupted image-text pairs</strong></summary>

```python
def image_hash(image):
    return hashlib.sha256(image.numpy().tobytes()).hexdigest()


# Build a deliberately contaminated copy without changing the clean experiment source.
contaminated_images = torch.cat([train_x, train_x[:80]], dim=0)
contaminated_image_labels = torch.cat([train_y, train_y[:80]], dim=0)
contaminated_caption_labels = contaminated_image_labels.clone()
corrupt_rows = torch.arange(0, 120, 4)
contaminated_caption_labels[corrupt_rows] = (contaminated_caption_labels[corrupt_rows] + 3) % 10
hashes = [image_hash(image) for image in contaminated_images]
hash_counts = Counter(hashes)

seen, keep = set(), []
for row, digest in enumerate(hashes):
    pair_matches = contaminated_image_labels[row] == contaminated_caption_labels[row]
    if digest not in seen and bool(pair_matches):
        seen.add(digest)
        keep.append(row)

audit_report = {
    "rows before": len(contaminated_images),
    "duplicate rows": sum(count - 1 for count in hash_counts.values()),
    "caption mismatches": int((contaminated_image_labels != contaminated_caption_labels).sum()),
    "rows after exact-dedup and pair audit": len(keep),
}
assert audit_report["duplicate rows"] == 80
assert audit_report["caption mismatches"] == len(corrupt_rows)
assert len(set(hashes[row] for row in keep)) == len(keep)
print(audit_report)
```

</details>

This audit can detect corruption because the controlled captions are generated from known labels. Real web pairs rarely provide such ground truth; quality models, human samples, source rules, and downstream ablations are needed. A filtering model should itself be versioned and evaluated for subgroup bias. Data governance also continues after release: deletion requests, newly discovered contamination, and changed legal or ethical constraints require lineage from source records to model versions.

### **Dense Models and Mixture-of-Experts** {#dense-models-mixture-of-experts}

A dense layer applies the same parameters to every token. A Mixture-of-Experts (MoE) layer contains many expert networks but routes each token to only the top $k$ experts. For router probabilities $p_e(h)$ and selected set $S(h)$,

$$
\operatorname{MoE}(h)=\sum_{e\in S(h)}p_e(h)E_e(h),\qquad |S(h)|=k\ll E.
$$

Total parameter capacity grows with the number of experts $E$, while active token-level computation grows mainly with $k$. This is conditional computation, not free scale. Expert weights must be stored and communicated; tokens can overflow expert capacity; the router can collapse onto a few experts; distributed all-to-all communication can dominate runtime. [Switch Transformer](https://arxiv.org/abs/2101.03961) simplified routing to top-1 experts and highlighted both efficiency and stability challenges.

![Top-k routing sends each token to a subset of experts before combining their outputs.](assets/dl13-moe-routing.svg){fig-align="center" width="76%" fig-alt="Token states flow through a router to selected experts and are combined, while an unused expert is skipped."}

*Original teaching diagram based on the sparse-routing mechanism in [Switch Transformer](https://arxiv.org/abs/2101.03961).* 

A common load-balancing auxiliary objective encourages the average router probability and realized token fraction to agree across experts. It does not guarantee semantic specialization. The small top-1 model below reports both total and approximately active expert parameters.

<details>
<summary><strong>PyTorch: compare dense and sparse expert classifiers</strong></summary>

```python
class DenseDigitModel(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.input = nn.Linear(64, hidden)
        self.block = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.head = nn.Linear(hidden, 10)

    def forward(self, images, return_aux=False):
        hidden = F.relu(self.input(images.flatten(1)))
        hidden = hidden + self.block(hidden)
        logits = self.head(F.relu(hidden))
        return (logits, logits.new_zeros(()), None) if return_aux else logits


class SparseMoEDigitModel(nn.Module):
    def __init__(self, hidden=48, experts=4):
        super().__init__()
        self.input = nn.Linear(64, hidden)
        self.router = nn.Linear(hidden, experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            for _ in range(experts)
        ])
        self.head = nn.Linear(hidden, 10)

    def forward(self, images, return_aux=False):
        hidden = F.relu(self.input(images.flatten(1)))
        probabilities = self.router(hidden).softmax(dim=-1)
        routes = probabilities.argmax(dim=-1)
        expert_output = torch.zeros_like(hidden)
        fractions = []
        for expert_index, expert in enumerate(self.experts):
            selected = routes == expert_index
            fractions.append(selected.float().mean())
            if selected.any():
                expert_output[selected] = expert(hidden[selected]) * probabilities[selected, expert_index].unsqueeze(1)
        fractions = torch.stack(fractions)
        mean_probability = probabilities.mean(dim=0)
        assignment_balance = len(self.experts) * torch.sum(fractions.detach() * mean_probability)
        probability_balance = len(self.experts) * torch.sum(mean_probability.square())
        balance_loss = assignment_balance + probability_balance
        logits = self.head(F.relu(hidden + expert_output))
        return (logits, balance_loss, fractions) if return_aux else logits


def train_routed_model(model, epochs=45):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits, auxiliary, _ = model(train_x, return_aux=True)
        (F.cross_entropy(logits, train_y) + 0.03 * auxiliary).backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        logits, _, fractions = model(test_x, return_aux=True)
    return float((logits.argmax(1) == test_y).float().mean()), fractions


seed_everything(1330)
dense_model, moe_model = DenseDigitModel(), SparseMoEDigitModel()
dense_accuracy, _ = train_routed_model(dense_model)
moe_accuracy, expert_fractions = train_routed_model(moe_model)
dense_parameters = sum(p.numel() for p in dense_model.parameters())
moe_parameters = sum(p.numel() for p in moe_model.parameters())
one_expert_parameters = sum(p.numel() for p in moe_model.experts[0].parameters())
active_proxy = moe_parameters - sum(p.numel() for p in moe_model.experts.parameters()) + one_expert_parameters

assert abs(float(expert_fractions.sum()) - 1.0) < 1e-5
print({
    "dense accuracy/parameters": (round(dense_accuracy, 3), dense_parameters),
    "MoE accuracy/total parameters": (round(moe_accuracy, 3), moe_parameters),
    "MoE approximate active parameters": active_proxy,
    "expert token fractions": [round(float(value), 3) for value in expert_fractions],
})
```

</details>

The toy router processes whole images rather than Transformer tokens, and `active_proxy` ignores communication and router cost. It demonstrates accounting, not production speed. In real MoE training, report expert utilization, dropped tokens, capacity factor, router entropy, all-to-all time, total parameters, and active FLOPs. A balanced router can still learn redundant experts; specialization must be tested rather than inferred from expert IDs.

### **Modality Encoders and Tokenization** {#modality-encoders-tokenization}

Multimodal systems must turn heterogeneous signals into computational units. Text uses subword or byte tokens; images use pixels, patches, regions, or learned visual tokens; audio uses waveform samples or time-frequency frames; video uses frames or space-time tubelets; structured records use typed fields and missingness indicators.

Tokenization is an information and cost decision. With image height $H$, width $W$, and patch size $P$, a non-overlapping patch encoder creates $T_I=HW/P^2$ tokens. A video with $F$ frames and temporal tubelet size $P_t$ creates approximately $T_V=FHW/(P_tP^2)$. Full self-attention memory grows as $O(T^2)$, so resolution and duration are systems parameters as well as data choices.

![Images, audio, video, and text require different sampling and token-budget contracts.](assets/dl13-modality-tokens.svg){fig-align="center" width="76%" fig-alt="An image is split into patches, audio into time frames, video into space-time tubelets, and text into subword tokens."}

*Original teaching diagram. The visual organization is informed by modality-specific tokenizers used in ViT-style models, wav2vec-style audio models, and [VideoMAE](https://arxiv.org/abs/2203.12602).* 

Each modality normally receives positional information and a modality/type embedding. Padding masks distinguish absent tokens from valid silence or black pixels. Time alignment and sampling rate must remain in metadata; otherwise two tensors with identical shapes can represent different physical intervals.

<details>
<summary><strong>PyTorch: trace image patches and text tokens into a shared width</strong></summary>

```python
def patchify(images, patch_size=2):
    patches = F.unfold(images, kernel_size=patch_size, stride=patch_size)
    return patches.transpose(1, 2)


shared_width = 32
image_patch_projection = nn.Linear(4, shared_width)
text_token_embedding = nn.Embedding(len(vocabulary), shared_width, padding_idx=0)
sample_patches = patchify(train_x[:5])
image_tokens = image_patch_projection(sample_patches)
text_tokens = text_token_embedding(train_text_ids[:5])

assert sample_patches.shape == (5, 16, 4)
assert image_tokens.shape == (5, 16, shared_width)
assert text_tokens.shape == (5, 7, shared_width)
assert train_text_mask[:5].shape == (5, 7)
print({
    "image token path": [(5, 1, 8, 8), tuple(sample_patches.shape), tuple(image_tokens.shape)],
    "text token path": [tuple(train_text_ids[:5].shape), tuple(text_tokens.shape)],
    "valid text tokens": train_text_mask[:5].sum(dim=1).tolist(),
})
```

</details>

A shared width does not imply shared semantics. Encoders can remain entirely separate and align only at projection heads, or modalities can enter a unified Transformer. Separate encoders preserve specialist inductive biases and make missing modalities easier to handle. Unified token streams permit rich interaction but require careful normalization, sampling, and capacity allocation so that a high-token-count modality does not dominate.

### **Early, Late, and Intermediate Fusion** {#early-late-intermediate-fusion}

Fusion answers **when modalities exchange information**.

- **Early fusion** combines raw or lightly encoded inputs. It can model low-level interactions but requires aligned sampling and tends to couple preprocessing tightly.
- **Intermediate fusion** lets specialist encoders produce features, then uses concatenation, gating, cross-attention, or a connector. This is the common compromise between modularity and interaction.
- **Late fusion** combines predictions or scores. It is robust to modular deployment and can calibrate each expert independently, but cannot learn fine token-level correspondence.

![Early, intermediate, and late fusion differ in when modality information interacts.](assets/dl13-fusion-taxonomy.svg){fig-align="center" width="76%" fig-alt="Three panels show modalities combined before encoding, through a middle connector, or after separate decisions."}

*Original comparison diagram synthesizing standard multimodal fusion patterns.*

The controlled task below predicts the digit from its image plus two supplied bits: even/odd and below/above five. Those attributes reduce ambiguity but do not identify all ten classes. This is a deliberate task definition, not accidental label leakage; the same attributes are available at inference. Missing-modality accuracy is measured by replacing the hint with zeros.

<details>
<summary><strong>PyTorch: compare early, intermediate, and late fusion</strong></summary>

```python
class FusionClassifier(nn.Module):
    def __init__(self, mode, hidden=48):
        super().__init__()
        self.mode = mode
        if mode == "early":
            self.joint = nn.Sequential(nn.Linear(66, hidden), nn.ReLU(), nn.Linear(hidden, 10))
        else:
            self.image_branch = nn.Sequential(nn.Linear(64, hidden), nn.ReLU())
            self.hint_branch = nn.Sequential(nn.Linear(2, 16), nn.ReLU())
            if mode == "intermediate":
                self.head = nn.Linear(hidden + 16, 10)
            elif mode == "late":
                self.image_head = nn.Linear(hidden, 10)
                self.hint_head = nn.Linear(16, 10)
            else:
                raise ValueError(mode)

    def forward(self, images, hints):
        flat = images.flatten(1)
        if self.mode == "early":
            return self.joint(torch.cat([flat, hints], dim=1))
        image_hidden = self.image_branch(flat)
        hint_hidden = self.hint_branch(hints)
        if self.mode == "intermediate":
            return self.head(torch.cat([image_hidden, hint_hidden], dim=1))
        return 0.75 * self.image_head(image_hidden) + 0.25 * self.hint_head(hint_hidden)


def train_fusion(model, epochs=45):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(train_x, train_hints), train_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        full = float((model(test_x, test_hints).argmax(1) == test_y).float().mean())
        missing = float((model(test_x, torch.zeros_like(test_hints)).argmax(1) == test_y).float().mean())
    return full, missing


fusion_models, fusion_results = {}, {}
for offset, mode in enumerate(("early", "intermediate", "late")):
    seed_everything(1340 + offset)
    fusion_models[mode] = FusionClassifier(mode)
    fusion_results[mode] = train_fusion(fusion_models[mode])

assert set(fusion_results) == {"early", "intermediate", "late"}
print({mode: {"all modalities": round(scores[0], 3), "hint missing": round(scores[1], 3)}
       for mode, scores in fusion_results.items()})
```

</details>

Fusion evaluation should include unimodal baselines, modality dropout, conflicting inputs, timestamp misalignment, and calibration. A multimodal model may ignore one modality if another predicts the training target more easily. Attention weights or larger ablation drops can suggest usage but do not prove causal grounding. The task must contain examples where each modality is necessary to test complementarity.

### **Cross-Attention and Shared Embedding Spaces** {#cross-attention-shared-embedding-spaces}

Shared embeddings and cross-attention solve related but different problems. A dual encoder maps each modality to one global vector and scores compatibility, commonly by cosine similarity:

$$
s(x,t)=\frac{f_I(x)^\top f_T(t)}{\tau},
$$

where normalized image and text embeddings share a dimension and $\tau$ is a temperature. Global vectors support efficient retrieval because each side can be indexed independently. They compress local detail.

Cross-attention retains token-level states. Image queries $Q_I$ can attend to text keys and values $K_T,V_T$:

$$
\operatorname{CrossAttn}(Q_I,K_T,V_T)=
\operatorname{softmax}\left(\frac{Q_IK_T^\top}{\sqrt{d_k}}+M\right)V_T.
$$

$M$ masks padding or invalid alignments. Cross-attention supports grounding and conditional reasoning but requires joint computation for each pair, making large-scale retrieval more expensive.

![Shared embedding alignment and cross-attention fusion support different operations.](assets/dl13-alignment-cross-attention.svg){fig-align="center" width="76%" fig-alt="The left panel aligns global image and text vectors for retrieval; the right connects image patch queries to text token keys and values for local interaction."}

*Original synthesis based on the dual-encoder objective in [CLIP](https://proceedings.mlr.press/v139/radford21a.html) and cross-attention connectors used by models such as [Flamingo](https://arxiv.org/abs/2204.14198).* 

Because many training images share the same class description, treating every off-diagonal pair as a negative would create false negatives. The loss below treats all pairs with the same digit label as positives.

<details>
<summary><strong>PyTorch: train a multi-positive image-text dual encoder and trace cross-attention</strong></summary>

```python
class DualEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.image_encoder = ImageEncoder(output_dim=embedding_dim)
        self.token_embedding = nn.Embedding(len(vocabulary), embedding_dim, padding_idx=0)
        self.text_projection = nn.Linear(embedding_dim, embedding_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.12)))

    def encode_image(self, images):
        return F.normalize(self.image_encoder(images), dim=-1)

    def encode_text(self, token_ids, token_mask):
        tokens = self.token_embedding(token_ids)
        mask = token_mask.unsqueeze(-1)
        pooled = (tokens * mask).sum(1) / mask.sum(1).clamp_min(1)
        return F.normalize(self.text_projection(pooled), dim=-1)

    def similarities(self, images, token_ids, token_mask):
        scale = self.logit_scale.exp().clamp(max=100)
        return scale * self.encode_image(images) @ self.encode_text(token_ids, token_mask).T


def multi_positive_loss(scores, labels):
    positives = labels[:, None] == labels[None, :]
    numerator_i = torch.logsumexp(scores.masked_fill(~positives, -1e9), dim=1)
    denominator_i = torch.logsumexp(scores, dim=1)
    numerator_t = torch.logsumexp(scores.T.masked_fill(~positives.T, -1e9), dim=1)
    denominator_t = torch.logsumexp(scores.T, dim=1)
    return 0.5 * ((denominator_i - numerator_i).mean() + (denominator_t - numerator_t).mean())


seed_everything(1350)
dual_encoder = DualEncoder()
optimizer = torch.optim.AdamW(dual_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(train_x, train_text_ids, train_text_mask, train_y),
                    batch_size=160, shuffle=True, generator=torch.Generator().manual_seed(1350))
for _ in range(45):
    dual_encoder.train()
    for batch_images, batch_ids, batch_mask, batch_labels in loader:
        optimizer.zero_grad()
        scores = dual_encoder.similarities(batch_images, batch_ids, batch_mask)
        loss = multi_positive_loss(scores, batch_labels)
        loss.backward()
        optimizer.step()

# Trace a cross-attention connector using the trained text-token embedding.
patch_projection = nn.Linear(4, 32)
cross_attention = nn.MultiheadAttention(32, num_heads=4, batch_first=True)
query_tokens = patch_projection(patchify(test_x[:4]))
key_value_tokens = dual_encoder.token_embedding(test_text_ids[:4])
fused_tokens, attention_weights = cross_attention(
    query_tokens, key_value_tokens, key_value_tokens,
    key_padding_mask=~test_text_mask[:4], need_weights=True,
)

assert fused_tokens.shape == (4, 16, 32)
assert attention_weights.shape == (4, 16, 7)
assert torch.isfinite(loss)
print({"final contrastive loss": round(float(loss.detach()), 3),
       "cross-attention output": tuple(fused_tokens.shape),
       "attention map": tuple(attention_weights.shape)})
```

</details>

The multi-positive mask uses labels because the chapter's text has only class-level semantics. A real image-caption corpus can contain multiple valid captions without class labels, and some apparent negatives may still describe the same concept. Batch composition therefore defines the contrastive task. Temperature, distributed negative gathering, duplicate handling, and caption specificity all change what geometry is learned.

### **Vision-Language Models** {#vision-language-models}

Vision-language models (VLMs) span several architectures:

- **Dual encoders** align independent image and text embeddings for retrieval and zero-shot classification, as in [CLIP](https://proceedings.mlr.press/v139/radford21a.html).
- **Fusion encoders** jointly process visual and textual tokens for matching, grounding, and discriminative question answering.
- **Vision-to-language generators** connect a visual encoder to a language decoder for captioning, VQA, and instruction following. [Flamingo](https://arxiv.org/abs/2204.14198) uses gated cross-attention to bridge pretrained vision and language components; [BLIP](https://arxiv.org/abs/2201.12086) combines understanding and generation objectives while addressing noisy captions.

Zero-shot CLIP-style classification turns class names into classifier weights. For class prompt $t_c$ and image $x$, predict

$$
\hat y=\arg\max_c f_I(x)^\top f_T(t_c).
$$

The classifier therefore depends on language wording. Prompt ensembling averages several normalized text embeddings per class, reducing sensitivity to one template. It does not create knowledge absent from pretraining.

<details>
<summary><strong>PyTorch: evaluate zero-shot classification and prompt sensitivity</strong></summary>

```python
@torch.no_grad()
def text_prototypes(template):
    texts = [template.format(name) for name in digit_names]
    token_ids, token_mask = encode_texts(texts)
    return dual_encoder.encode_text(token_ids, token_mask)


@torch.no_grad()
def zero_shot_score(images, labels, prototypes):
    image_embeddings = dual_encoder.encode_image(images)
    predictions = (image_embeddings @ prototypes.T).argmax(1)
    return float((predictions == labels).float().mean())


dual_encoder.eval()
template_prototypes = [text_prototypes(template) for template in caption_templates]
template_scores = [zero_shot_score(test_x, test_y, prototypes) for prototypes in template_prototypes]
ensemble_prototypes = F.normalize(torch.stack(template_prototypes).mean(dim=0), dim=-1)
ensemble_score = zero_shot_score(test_x, test_y, ensemble_prototypes)

assert ensemble_prototypes.shape == (10, 32)
print({
    "per-template zero-shot accuracy": [round(score, 3) for score in template_scores],
    "prompt ensemble accuracy": round(ensemble_score, 3),
})
```

</details>

VLM output can be linguistically plausible while visually unsupported. Evaluation must separate object recognition, OCR, spatial reasoning, counting, temporal understanding, and hallucination. Dataset overlap and prompt-format sensitivity can inflate apparent zero-shot ability. Grounding tests should change the image while holding the question fixed, mask relevant regions, and include answerable/unanswerable pairs. A fluent explanation is not evidence that the model used the correct visual region.

### **Audio, Video, and Multimodal Sequence Models** {#audio-video-multimodal-sequence-models}

Audio and video add time. Audio representations must preserve sampling rate, channel layout, window size, and hop length. Video representations must choose frame rate, spatial resolution, clip duration, and temporal tubelet size. A ten-second event can be represented by tens of text tokens, hundreds of audio frames, or thousands of video patches; naïvely concatenating all tokens creates severe imbalance.

Several architectural strategies manage long multimodal streams:

- local or factorized spatial-temporal attention;
- temporal pooling, striding, learned resampling, or latent bottlenecks;
- modality-specific encoders followed by sparse cross-attention;
- memory, recurrence, or retrieval over long recordings;
- masked prediction at high masking ratios, as explored in [VideoMAE](https://arxiv.org/abs/2203.12602).

[ImageBind](https://arxiv.org/abs/2305.05665) aligns image, text, audio, depth, thermal, and IMU representations in one space, using image-paired data as a binding signal. This illustrates an important idea: every modality pair need not have direct supervision. It also creates a hub-modality risk; concepts poorly represented in the image anchor may align weakly.

<details>
<summary><strong>Python: compare modality token budgets before choosing attention</strong></summary>

```python
def modality_budget(image_hw=(224, 224), image_patch=16, audio_seconds=10,
                    audio_hop_ms=20, video_frames=32, tubelet=2):
    image_tokens = (image_hw[0] // image_patch) * (image_hw[1] // image_patch)
    audio_tokens = int(audio_seconds * 1000 / audio_hop_ms)
    video_tokens = (video_frames // tubelet) * image_tokens
    total_tokens = image_tokens + audio_tokens + video_tokens
    return {
        "image tokens": image_tokens,
        "audio frame tokens": audio_tokens,
        "video tubelet tokens": video_tokens,
        "concatenated tokens": total_tokens,
        "full-attention score entries": total_tokens ** 2,
    }


token_budget = modality_budget()
pooled_budget = modality_budget(image_hw=(112, 112), image_patch=16,
                                audio_seconds=10, audio_hop_ms=40,
                                video_frames=16, tubelet=4)
assert pooled_budget["full-attention score entries"] < token_budget["full-attention score entries"]
print({"high-resolution": token_budget, "pooled/strided": pooled_budget})
```

</details>

Token count is not information content. Aggressive pooling may erase short sounds, text in video, or brief actions. Uniform frame sampling can miss rare events; voice activity detection can remove meaningful silence; asynchronous sensors require timestamps rather than assumed index alignment. Evaluation should stratify by duration and event frequency and should test temporal shuffling, modality delay, and missing segments.

### **Multimodal Pretraining Objectives** {#multimodal-pretraining-objectives}

Different objectives teach different relationships:

- **Contrastive alignment** pulls matched global representations together and supports retrieval.
- **Image-text matching** classifies whether a pair belongs together and can model harder interactions through a fusion encoder.
- **Masked modeling** predicts hidden image patches, audio frames, video tubelets, or text tokens from context.
- **Autoregressive generation** predicts the next token or modality output conditioned on preceding context.
- **Distillation and pseudo-labeling** transfer a specialist or teacher into another modality or unified student.

Joint training often combines losses,

$$
\mathcal{L}=\lambda_{\text{contrast}}\mathcal{L}_{\text{contrast}}
+\lambda_{\text{match}}\mathcal{L}_{\text{match}}
+\lambda_{\text{mask}}\mathcal{L}_{\text{mask}}
+\lambda_{\text{gen}}\mathcal{L}_{\text{gen}}.
$$

The coefficients determine gradient scale and representation priorities. Adding objectives can create interference rather than free capability. [BLIP](https://arxiv.org/abs/2201.12086) is a useful example of combining understanding and generation while filtering noisy captions; [FLIP](https://arxiv.org/abs/2212.00794) uses image masking to improve language-image pretraining efficiency.

The code below keeps the trained dual encoder fixed, learns an image-text matching head with explicit negative pairs, and separately trains a masked-pixel reconstructor. These are small objective probes, not a unified production model.

<details>
<summary><strong>PyTorch: probe matching and masked reconstruction objectives</strong></summary>

```python
with torch.no_grad():
    image_features = dual_encoder.encode_image(train_x[:300])
    text_features = dual_encoder.encode_text(train_text_ids[:300], train_text_mask[:300])
labels_300 = train_y[:300]
negative_rows = []
for row, label in enumerate(labels_300):
    candidate = (row + 1) % len(labels_300)
    while labels_300[candidate] == label:
        candidate = (candidate + 1) % len(labels_300)
    negative_rows.append(candidate)
negative_rows = torch.tensor(negative_rows)

def pair_features(left, right):
    # Matching needs an interaction term; concatenated marginals alone cannot
    # express whether two vectors agree with a single linear decision layer.
    return torch.cat([left * right, (left - right).abs()], dim=1)


positive_pairs = pair_features(image_features, text_features)
negative_pairs = pair_features(image_features, text_features[negative_rows])
match_inputs = torch.cat([positive_pairs, negative_pairs], dim=0)
match_targets = torch.cat([torch.ones(300), torch.zeros(300)]).long()
match_train_rows = torch.cat([torch.arange(240), torch.arange(300, 540)])
match_eval_rows = torch.cat([torch.arange(240, 300), torch.arange(540, 600)])
matching_head = nn.Linear(64, 2)
optimizer = torch.optim.AdamW(matching_head.parameters(), lr=1e-2)
for _ in range(80):
    optimizer.zero_grad()
    matching_loss = F.cross_entropy(
        matching_head(match_inputs[match_train_rows]), match_targets[match_train_rows]
    )
    matching_loss.backward()
    optimizer.step()
matching_accuracy = float((
    matching_head(match_inputs[match_eval_rows]).argmax(1) == match_targets[match_eval_rows]
).float().mean())

seed_everything(1370)
mask = torch.rand_like(train_x[:300]) < 0.35
masked_images = train_x[:300].masked_fill(mask, 0.0)
reconstructor = nn.Sequential(nn.Flatten(), nn.Linear(64, 96), nn.ReLU(), nn.Linear(96, 64))
optimizer = torch.optim.AdamW(reconstructor.parameters(), lr=3e-3)
for _ in range(80):
    optimizer.zero_grad()
    reconstruction = reconstructor(masked_images).view_as(masked_images)
    masked_loss = F.mse_loss(reconstruction[mask], train_x[:300][mask])
    masked_loss.backward()
    optimizer.step()

assert matching_accuracy > 0.5
assert torch.isfinite(masked_loss)
print({"matching accuracy on held-out pairs": round(matching_accuracy, 3),
       "masked-pixel reconstruction MSE": round(float(masked_loss.detach()), 4)})
```

</details>

Negative construction is part of the matching task: easy random negatives may reward superficial mismatch, while hard negatives can accidentally be valid pairs. Reconstruction quality can prioritize texture over semantics. Objective evaluation should therefore include transfer probes, retrieval, grounding, generation, and ablations rather than assuming the pretraining loss captures all desired capabilities.

### **Adaptation, Evaluation, and Capability Transfer** {#adaptation-evaluation-capability-transfer}

Foundation-model evaluation is a matrix over capabilities, modalities, tasks, domains, and adaptation budgets. At minimum, distinguish:

- **zero-shot transfer:** no target parameter update;
- **few-shot or linear probing:** small labeled target evidence over frozen representations;
- **PEFT/full tuning:** increasing update capacity, as studied in Chapter 12;
- **retrieval and grounding:** whether paired concepts and local evidence align;
- **robustness and modality ablation:** whether behavior survives shift, missing inputs, or conflict.

For class-level retrieval, Recall@$k$ asks whether a valid class text appears among the top $k$ text scores. A linear probe asks whether a frozen representation exposes a target boundary. Neither proves generative faithfulness. Prompted generation additionally needs factuality, calibration, refusal behavior, and human evaluation under a specified rubric.

The next audit uses the shared embedding for zero-shot digit recognition, class retrieval, a parity probe, and a shifted-image stress test. The shift matches the label-preserving corruption family used in Chapter 12.

<details>
<summary><strong>Python: evaluate transfer, retrieval, shift, and missing modalities</strong></summary>

```python
def shifted_domain(images, seed=1380):
    shifted = torch.zeros_like(images)
    shifted[:, :, :, 1:] = images[:, :, :, :-1]
    blurred = F.avg_pool2d(shifted, 3, stride=1, padding=1)
    generator = torch.Generator().manual_seed(seed)
    noise = 0.05 * torch.randn(images.shape, generator=generator)
    return (0.84 * blurred + noise).clamp(0.0, 1.0)


dual_encoder.eval()
with torch.no_grad():
    train_embeddings = dual_encoder.encode_image(train_x).numpy()
    test_embeddings = dual_encoder.encode_image(test_x).numpy()
    test_image_embeddings = torch.tensor(test_embeddings)
    similarities = test_image_embeddings @ ensemble_prototypes.T
    ranking = similarities.argsort(dim=1, descending=True)
    recall_at_1 = float((ranking[:, :1] == test_y[:, None]).any(dim=1).float().mean())
    recall_at_5 = float((ranking[:, :5] == test_y[:, None]).any(dim=1).float().mean())
    shifted_accuracy = zero_shot_score(shifted_domain(test_x), test_y, ensemble_prototypes)

parity_probe = LogisticRegression(max_iter=500, random_state=1380)
parity_probe.fit(train_embeddings, (train_y.numpy() % 2))
parity_accuracy = accuracy_score(test_y.numpy() % 2, parity_probe.predict(test_embeddings))

intermediate_model = fusion_models["intermediate"]
intermediate_model.eval()
with torch.no_grad():
    missing_hint_accuracy = float((
        intermediate_model(test_x, torch.zeros_like(test_hints)).argmax(1) == test_y
    ).float().mean())

assert recall_at_5 >= recall_at_1
print({
    "zero-shot / retrieval R@1": round(recall_at_1, 3),
    "retrieval R@5": round(recall_at_5, 3),
    "frozen embedding parity probe": round(float(parity_accuracy), 3),
    "shifted zero-shot accuracy": round(shifted_accuracy, 3),
    "intermediate fusion with hint missing": round(missing_hint_accuracy, 3),
})
```

</details>

The class-level text makes zero-shot classification and R@1 numerically equivalent here; real retrieval has many captions and many valid images, so image-to-text and text-to-image recall differ. Capability transfer should be compared against task-specific baselines and uncertainty intervals. If a model is selected on the same benchmark repeatedly, the benchmark becomes part of development and needs a fresh final test.

### **The Foundation Model Lifecycle** {#foundation-model-lifecycle}

A foundation-model lifecycle is a set of versioned artifacts and gates:

1. **Problem and policy definition:** intended uses, excluded uses, risk tolerance, data rights, and success criteria.
2. **Data pipeline:** provenance, mixture weights, filtering, deduplication, deletion lineage, and held-out evaluation boundaries.
3. **Pretraining:** architecture, objective, compute, checkpoints, scaling evidence, and incident logs.
4. **Adaptation:** prompt, retrieval, PEFT, full tuning, tools, and use-specific safety controls.
5. **Evaluation and release:** capability, robustness, safety, privacy, security, subgroup, and systems evidence.
6. **Monitoring and revision:** drift, abuse, feedback quality, rollback, deprecation, and model/data version links.

Model cards and data documentation do not replace evaluation, but they make assumptions discoverable. Release decisions should identify which evidence applies to the base and which applies only to one adapted system. An API update, changed safety policy, or new retrieval corpus can change behavior even if the model identifier remains visually similar.

<details>
<summary><strong>Python: create an auditable evaluation record and drift signal</strong></summary>

```python
with torch.no_grad():
    clean_embeddings = dual_encoder.encode_image(test_x)
    shifted_embeddings = dual_encoder.encode_image(shifted_domain(test_x, seed=1390))
    paired_cosine = F.cosine_similarity(clean_embeddings, shifted_embeddings, dim=1)
    embedding_drift = float(1.0 - paired_cosine.mean())

evaluation_record = {
    "model": {"family": "tiny dual encoder", "version": "dl13-demo-v1"},
    "data": {
        "source": "UCI Optical Recognition of Handwritten Digits via scikit-learn",
        "doi": "10.24432/C50P49",
        "license": "CC BY 4.0",
        "constructed_text": True,
        "split_seed": 1313,
    },
    "evidence": {
        "prompt_ensemble_zero_shot": round(ensemble_score, 3),
        "shifted_zero_shot": round(shifted_accuracy, 3),
        "mean_embedding_drift": round(embedding_drift, 3),
        "missing_hint_accuracy": round(missing_hint_accuracy, 3),
    },
    "limitations": [
        "class-level synthetic captions",
        "ten classes and small images",
        "mechanism demonstration rather than a VLM benchmark",
    ],
}
assert evaluation_record["data"]["constructed_text"] is True
assert 0.0 <= embedding_drift <= 2.0
print(evaluation_record)
```

</details>

A production monitor needs thresholds fitted to normal variation, not arbitrary constants. Embedding drift can detect change without identifying harm; unchanged embeddings can coexist with changed decoder behavior or policy. Monitor input distributions, retrieval sources, output quality, safety events, latency, and human escalation together. Preserve enough lineage to reproduce the exact base, adapter, prompt, index, and policy active during an incident.

### **From Pretraining to Post-Training** {#pretraining-to-post-training}

Pretraining learns broad statistical structure from large corpora. Continued pretraining may specialize the base to a domain or time period. Supervised or instruction tuning teaches target response formats and task following. Preference optimization, reinforcement learning, critique, rejection sampling, and safety tuning then shape behavior using comparisons, rewards, or policy constraints. Serving adds system prompts, retrieval, tools, filters, and monitoring.

These stages solve different problems. Pretraining loss rewards prediction of corpus regularities, not truthfulness or user intent. Instruction tuning can improve interface compliance without adding reliable world knowledge. Preference alignment can shift style and trade off capabilities. Retrieval can update evidence without changing parameters but introduces source quality and prompt-injection risk.

A useful dependency graph is

```text
curated pretraining data
        -> base representations and capabilities
        -> continued pretraining or adaptation
        -> instruction/task tuning
        -> preference and safety alignment
        -> retrieval/tools/policies at serving time
        -> monitored deployment evidence
```

Evaluation should be repeated after every behavior-changing stage. Base-model benchmarks cannot certify the final application, and final application tests do not reveal all upstream capability or privacy risks. Chapter 18 develops post-training objectives such as supervised fine-tuning, reward modeling, PPO, and DPO; the present chapter keeps only their position in the lifecycle.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Foundation models organize deep learning around reusable upstream assets and many downstream adaptations. Multimodal models extend those assets across signals with different sampling rates, token budgets, noise processes, and semantics. Their design is therefore wider than choosing a Transformer block.

| Design question | Main options | What must be measured |
|---|---|---|
| Scale allocation | parameters, data, steps, compute | held-out loss, uncertainty, inference cost |
| Capacity | dense or sparse MoE | total/active parameters, FLOPs, routing and communication |
| Modality interface | separate encoders or unified tokens | information retained, masks, token balance |
| Fusion | early, intermediate, late | complementarity, missing/conflicting modalities |
| Alignment | shared embeddings or token-level cross-attention | retrieval, grounding, pairwise compute |
| Objectives | contrastive, matching, masked, generative | transfer, interference, shortcut behavior |
| Adaptation | prompting, retrieval, PEFT, full tuning | target quality, retention, memory, latency |
| Governance | documentation, release gates, monitoring | lineage, subgroup risk, incidents, rollback |

The chapter's small image-text experiment exposed several general lessons. Captions and negative sampling define the contrastive task. A shared embedding supports cheap retrieval but compresses local evidence. Fusion can ignore a modality unless the task requires it. Sparse experts separate total capacity from active computation but add routing and communication failure modes. Scaling curves are empirical within a regime, and data curation can change results as much as architecture.

The practical standard is an auditable chain: document data provenance and constructed modalities; state the token and compute budget; compare dense, sparse, alignment, and fusion choices under controlled evidence; evaluate zero-shot, adapted, shifted, and missing-modality behavior; and link every deployment to exact model, data, retrieval, prompt, and policy versions.

Chapter 14 turns to generative modeling itself. It explains probability models, maximum likelihood, autoregressive factorization, exposure bias, and decoding. Those mechanisms are prerequisites for understanding the generative interfaces that many foundation models expose, but they are not required for a model to serve as a reusable foundation.